In [ ]:
import zipfile
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

LOAD DATA FROM KAGGLE

In [ ]:
from google.colab import files
files.upload()


In [ ]:
import os

!mkdir -p ~/.kaggle # Create the hidden .kaggle directory in the user's home folder if it doesn't exist
!mv kaggle.json ~/.kaggle/ # Move the uploaded API token credentials file to the required .kaggle directory
!chmod 600 ~/.kaggle/kaggle.json # Update file permissions so only the owner can read/write to the credentials file

In [ ]:
!pip install -q kaggle

dataset = "chethuhn/network-intrusion-dataset"
download_path = "./IDS2"

!mkdir -p {download_path} # Create the target download folder if it does not already exist
!kaggle datasets download -d {dataset} -p {download_path}

EXTRACT FROM ZIP FILE

In [ ]:
# List files
print(os.listdir(download_path))

# Automatically find zip file
zip_file = [f for f in os.listdir(download_path) if f.endswith(".zip")][0]

# Extract
with zipfile.ZipFile(os.path.join(download_path, zip_file), 'r') as zip_ref:
    zip_ref.extractall(download_path)

print("After extraction:", os.listdir(download_path))

LOAD DATA INTO PANDAS DATAFRAME

In [ ]:
dfs = []
for root, dirs, files in os.walk(download_path):
  for file in files:
        if file.endswith(".csv"):
            csv_file = os.path.join(root, file)
            df_temp = pd.read_csv(csv_file)
            #df_temp = df_temp.sample(n=50000, random_state=42)
            dfs.append(df_temp)
df = pd.concat(dfs, axis=0)


DATA PREPROCESSING

In [ ]:
# 1. Clean hidden leading/trailing spaces from all column names
df.columns = df.columns.str.strip()

print("shape: ")
print(df.shape)

print("columns: ")
print(df.columns)

print("info: ")
print(df.info())

print("nulls: ")
print(df.isnull().sum())

print("labels: ")
print(df['Label'].value_counts())  # adjust column name if needed





* Missing values in Flow Bytes (Line 14): Non-Null Count: 225,741
* Duplicate Feature "Fwd Header Length" (Line 34 and 55)
* INF and -INF values




In [ ]:
# Replace infinite values
df = df.replace([np.inf, -np.inf], np.nan)
#drop null content
print(df.isnull().sum().sum())  # check first
df = df.dropna()
#drop duplicate columns
df = df.drop_duplicates(keep='first')
#Encode labels
le = LabelEncoder()
df['Label'] = le.fit_transform(df['Label'])
# split the col label from other features
x = df.drop('Label', axis=1)
y = df['Label']
#print(df.info())
print(df['Label'].value_counts())

In [ ]:
"""print("Shape:", df.shape)
print("Nulls:", df.isnull().sum().sum())
print("Infinite:", np.isinf(df.values).sum())
print("Duplicates:", df.duplicated().sum())

print("\nLabel Mapping:")
for i, label in enumerate(le.classes_):
    print(f"{label} → {i}")

print("\nClass Distribution:")
print(df['Label'].value_counts())

print("\nFeature Shape:", x.shape)
print("Label Shape:", y.shape)"""

TRAIN/TEST SPLIT

In [ ]:
# train/test split
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

#scale features
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

class_weights = compute_sample_weight(class_weight='balanced', y=y_train)

TRAINING TWO MODELS:
* FIRST ON RANDOM FOREST
* SECOND ON XGBOOST

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model1 = RandomForestClassifier(class_weight='balanced')
model1.fit(x_train, y_train)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
y_pred1 = model1.predict(x_test)
print("Random Forest:\n", classification_report(y_test, y_pred1))
print(confusion_matrix(y_test, y_pred1))

In [ ]:
from xgboost import XGBClassifier

model2 = XGBClassifier(
    objective='multi:softprob',
    num_class=len(np.unique(y_train)),
    eval_metric='mlogloss',
    use_label_encoder=False
)
model2.fit(x_train, y_train, sample_weight=class_weights)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
y_pred2 = model2.predict(x_test)
print("XGboost:\n", classification_report(y_test, y_pred2))
print(confusion_matrix(y_test, y_pred2))

FINDING BEST MODEL

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

models = {"RandomForest": model1, "XGBoost": model2}
trained = {}
for name, model in models.items():
    trained[name] = model

best_model = None
best_f1 = 0

for name, model in trained.items():

    y_pred = model.predict(x_test)

    f1 = f1_score(y_test, y_pred, average='macro')

    print("\n", name)
    print("Macro F1:", f1)
    #print(classification_report(y_test, y_pred))
    print(classification_report(y_test, y_pred, target_names=le.classes_))

    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(8,6))
    #sns.heatmap(cm, cmap="Blues")
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=le.classes_,
        yticklabels=le.classes_
    )
    plt.title(name + " Confusion Matrix")
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    if f1 > best_f1:
        best_f1 = f1
        best_model = name

print("\nBest Model:", best_model)

SAVE MODEL

In [ ]:
import joblib
joblib.dump(trained[best_model], "best_model.pkl")
joblib.dump(le, "label_encoder.pkl")
joblib.dump(scaler, "scaler.pkl")

TEST MODEL

In [ ]:
def IDS_predict(sample_index):
    model = trained[best_model]

    # 1. Grab single row (already scaled)
    sample = x_test[sample_index].reshape(1, -1)

    # 2. Get prediction scalar
    pred = model.predict(sample)[0]
    predicted_label = le.inverse_transform([pred])[0]

    # 3. Get true scalar label safely
    true_numeric = y_test.iloc[sample_index] if hasattr(y_test, "iloc") else y_test[sample_index]
    true_label = le.inverse_transform([true_numeric])[0]

    print(f"\n--- Sample #{sample_index} Evaluation ---")
    print(f"Predicted: {predicted_label}")
    print(f"Actual:    {true_label}")

    # 4. Accuracy & Alert logic
    if pred == true_numeric:
        print("Result: CORRECT")
    else:
        print("Result: MISCLASSIFIED")

    if "BENIGN" in predicted_label:
        print("Traffic allowed")
    else:
        print(f"Alert triggered: {predicted_label}")


In [ ]:
import random

for i in random.sample(range(len(x_test)), 5):
    IDS_predict(i)